# Dammam Airport Traffic: Exploration and Cleaning

Official domestic and international monthly traffic, January 2024–October 2025. Run cells in order using the project `.venv` kernel, with `notebooks` as the working folder.

In [1]:
import pandas as pd

## 1. Inspect the domestic file

Skip the three introductory lines. Latin-1 lets us inspect the file; known city text artifacts are corrected below.

In [2]:
file = "../data/raw/2024-2025-Open_Data_Domestic.csv"
data = pd.read_csv(file, skiprows=3, encoding="latin-1"
)

In [3]:
data = data.dropna(axis=1, how="all")
data.columns = data.columns.str.strip()
data.head()

,Airport IATA,Year,Type,Month,Arrival/Departure,Destination_City,Pax,ATMs
0,DMM,2025.0,Domestic,Jan,Arrival,Abhaÿ,"20,359",132
1,DMM,2025.0,Domestic,Feb,Arrival,Abhaÿ,"17,561",117
2,DMM,2025.0,Domestic,Mar,Arrival,Abhaÿ,"15,262",122
3,DMM,2025.0,Domestic,Apr,Arrival,Abhaÿ,"23,333",150
4,DMM,2025.0,Domestic,May,Arrival,Abhaÿ,"22,267",162


In [4]:
data["Arrival/Departure"].value_counts(dropna=False)

Arrival/Departure
Arrival      337
Departue     337
NaN            1
Name: count, dtype: int64

In [5]:
data[data["Arrival/Departure"].isna()]

,Airport IATA,Year,Type,Month,Arrival/Departure,Destination_City,Pax,ATMs
674,NaN,NaN,NaN,NaN,NaN,NaN,"11,557,402","83,949"


## 2. Separate totals and clean domestic values

Keep the published total row for validation, outside the traffic records.

In [6]:
source_total = data[data["Arrival/Departure"].isna()].copy()

In [7]:
data = data[data["Arrival/Departure"].notna()].copy()

In [8]:
data["Arrival/Departure"] = (
    data["Arrival/Departure"]
    .str.strip()
    .replace({"Departue": "Departure"})
)

In [9]:
data["Destination_City"].unique()

array(['Abhaÿ', 'Al-Baha', 'AlQassim', 'Alula', 'Bisha', 'Hail', 'Jeddah',
       'Jizan ', 'Madinah', 'Najran', 'Neom', 'Red Sea', 'Riyadh',
       'Tabuk', 'Taif', 'Yanbu', 'Najran ', 'Hail ', 'Neom ', 'Riyadh ',
       'Taif ', 'Tabuk ', 'Alula ', 'Yanbu ', 'AlQaisumah'], dtype=object)

In [10]:
data["Destination_City"] = data["Destination_City"].str.strip().replace({"Abhaÿ": "Abha"})

In [11]:
data["Destination_City"].value_counts(dropna=False)

Destination_City
Abha          44
Al-Baha       44
AlQassim      44
Alula         44
Bisha         44
Hail          44
Jeddah        44
Jizan         44
Madinah       44
Najran        44
Neom          44
Riyadh        44
Tabuk         44
Yanbu         44
Taif          44
Red Sea       10
AlQaisumah     4
Name: count, dtype: int64

In [12]:
data[["Year", "Pax", "ATMs"]].dtypes

Year    float64
Pax      object
ATMs     object
dtype: object

In [13]:
data["Year"] = data["Year"].astype("int64")

In [14]:
data = data.rename(columns={
    "Pax": "Passengers",
    "ATMs": "Flights",
    "Airport IATA":"Airport"

})

In [15]:
data["Passengers"] = data["Passengers"].str.strip().str.replace(",","", regex=False)
data["Passengers"] = data["Passengers"].astype("int64")
data["Flights"] =  data["Flights"].str.strip() 
data["Flights"] =  data["Flights"].astype("int64")

In [16]:
data = data.rename(columns={
    "Airport": "airport_code",
    "Year": "year",
    "Type": "flight_type",
    "Month": "month",
    "Arrival/Departure": "direction",
    "Destination_City": "city",
    "Passengers": "passengers",
    "Flights": "flights"
})

data.columns.tolist()

['airport_code',
 'year',
 'flight_type',
 'month',
 'direction',
 'city',
 'passengers',
 'flights']

In [17]:
for column in ["airport_code", "month", "flight_type"]:
    print(column)
    print(data[column].value_counts(dropna=False))

airport_code
airport_code
DMM    674
Name: count, dtype: int64
month
month
Jan    64
Feb    62
Mar    62
Apr    62
May    62
Jun    60
Jul    60
Aug    60
Sep    60
Oct    60
Dec    32
Nov    30
Name: count, dtype: int64
flight_type
flight_type
Domestic    674
Name: count, dtype: int64


## 3. Add dates and validate domestic traffic

The first day represents the reporting month, not an individual flight date.

In [18]:
data["date"] = pd.to_datetime(
    data["year"].astype(str) + "-" + data["month"],
    format="%Y-%b"
)

In [19]:
month_mapping = {
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12
}

data["month_number"] = data["month"].map(month_mapping)

In [20]:
data["quarter"] = data["date"].dt.quarter

In [21]:
data = data[
    [
        "airport_code",
        "date",
        "year",
        "quarter",
        "month_number",
        "month",
        "flight_type",
        "direction",
        "city",
        "passengers",
        "flights",
    ]
]

In [22]:
source_total["Pax"] = (
    source_total["Pax"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .astype("int64")
)

source_total["ATMs"] = (
    source_total["ATMs"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .astype("int64")
)

print("Calculated passengers:", data["passengers"].sum())
print("Source passengers:", source_total["Pax"].iloc[0])

print("Calculated flights:", data["flights"].sum())
print("Source flights:", source_total["ATMs"].iloc[0])

Calculated passengers: 11557402
Source passengers: 11557402
Calculated flights: 83949
Source flights: 83949


In [23]:
assert data["passengers"].sum() == source_total["Pax"].iloc[0]
assert data["flights"].sum() == source_total["ATMs"].iloc[0]

In [24]:
key_columns = [
    "airport_code",
    "year",
    "month_number",
    "flight_type",
    "direction",
    "city",
]

assert data.shape == (674, 11)
assert data.isna().sum().sum() == 0
assert data.duplicated(subset=key_columns).sum() == 0
assert (data["passengers"] > 0).all()
assert (data["flights"] > 0).all()

assert set(data["direction"]) == {"Arrival", "Departure"}
assert set(data["airport_code"]) == {"DMM"}
assert set(data["flight_type"]) == {"Domestic"}

print("All validation checks passed.")

All validation checks passed.


## 4. Inspect and clean international traffic

Apply the same structure checks, then inspect city spellings and missing counts.

In [25]:
international_file = "../data/raw/2024-2025-Open_Data_International.csv"

international = pd.read_csv(
    international_file,
    skiprows=3,
    encoding="latin-1"
)

In [26]:
international = international.dropna(axis=1, how="all")
international.columns = international.columns.str.strip()

In [27]:
international_source_total = international[
    international["Arrival/Departure"].isna()
].copy()

international = international[
    international["Arrival/Departure"].notna()
].copy()

In [28]:
international["Arrival/Departure"] = (
    international["Arrival/Departure"].str.strip()
)

In [29]:
international["Arrival/Departure"].value_counts(dropna=False)

Arrival/Departure
Departure    906
Arrival      899
Name: count, dtype: int64

In [30]:
international["Destination_City"] = international["Destination_City"].str.strip()
international["Destination_City"].value_counts(dropna=False)

Destination_City
Abu Dhabi             44
Addis Ababa           44
Alexandria            44
Amman                 44
Amsterdam             44
Bahrain               44
Baku                  44
Beirut                44
Mumbai                44
Calicut               44
Cairo                 44
Delhi                 44
Colombo               44
Frankfurt             44
Dubai                 44
Dhaka                 44
Doha                  44
Hyderabad             44
Karachi               44
Istanbul              44
Islamabad             44
Najaf                 44
Muscat                44
Sharjah               44
Kathmandu             44
Kuwait                44
Lahore                44
Lucknow               44
Mangalore             44
Sialkot               44
Thiruvananthapuram    44
Port Sudan            44
Sohag                 36
Kannur                34
Chennai               32
Multan                26
kochi                 24
Mashhad               22
Bangalore             20
Kochi   

In [31]:
for city in sorted(international["Destination_City"].unique()):
    print(repr(city))

'Abu Dhabi'
'Addis Ababa'
'Alexandria'
'Amman'
'Amsterdam'
'Asyut'
'Bagdad'
'Baghdad'
'Bahrain'
'Baku'
'Bangalore'
'Beijingÿ'
'Beirut'
'Budapest'
'Cairo'
'Calicut'
'Chennai'
'Colombo'
'Damascus'
'Delhi'
'Dhaka'
'Doha'
'Dubai'
'Frankfurt'
'Hyderabad'
'Islamabad'
'Istanbul'
'Kannur'
'Karachi'
'Kathmandu'
'Kochi'
'Kuwait'
'Lahore'
'Lucknow'
'Mangalore'
'Manila'
'Mashhad'
'Multan'
'Mumbai'
'Muscat'
'Najaf'
'Port Sudan'
'Rome'
'Salalah'
'Sharjah'
'Sialkot'
'Sohag'
'Tbilisi'
'Thiruvananthapuram'
'Tiruchirapally'
'Trabzon'
'Vienna'
'kochi'


In [32]:
international["Destination_City"] = (
    international["Destination_City"]
    .replace({
        "Bagdad": "Baghdad",
        "Beijingÿ": "Beijing"
        
    })
)

In [33]:
international = international.rename(columns={
    "Airport IATA": "airport_code",
    "Year": "year",
    "Type": "flight_type",
    "Month": "month",
    "Arrival/Departure": "direction",
    "Destination_City": "city",
    "Pax": "passengers",
    "ATMs": "flights"
})

In [34]:
international["city"] = international["city"].replace({
    "kochi": "Kochi"
})

In [35]:
international.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1805 entries, 0 to 1804
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   airport_code  1805 non-null   object 
 1   year          1805 non-null   float64
 2   flight_type   1805 non-null   object 
 3   month         1805 non-null   object 
 4   direction     1805 non-null   object 
 5   city          1805 non-null   object 
 6   passengers    1805 non-null   object 
 7   flights       1805 non-null   object 
dtypes: float64(1), object(7)
memory usage: 126.9+ KB


## 5. Convert numbers and inspect missing passengers

The source uses `-` for 16 passenger values. Keep these missing; do not replace them with zero.

In [36]:
international["year"] = international["year"].astype("int64")

international["passengers"] = (
    international["passengers"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .replace("-", pd.NA)
    .astype("Int64")
)

international["flights"] = (
    international["flights"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .replace("-", pd.NA)
    .astype("int64")
)

In [37]:
international.loc[international["passengers"].isna(),["year", "month", "direction", "city", "passengers", "flights"]]

,year,month,direction,city,passengers,flights
754,2025,Mar,Departure,Baghdad,<NA>,2
755,2025,Apr,Departure,Baghdad,<NA>,2
759,2025,Aug,Departure,Baghdad,<NA>,2
760,2025,Sep,Departure,Baghdad,<NA>,1
982,2024,Jan,Arrival,Baghdad,<NA>,7
983,2024,Feb,Arrival,Baghdad,<NA>,7
984,2024,Mar,Arrival,Baghdad,<NA>,2
987,2024,Jun,Arrival,Baghdad,<NA>,12
988,2024,Jul,Arrival,Baghdad,<NA>,3
989,2024,Aug,Arrival,Baghdad,<NA>,3


In [38]:
print("Missing passenger values:", international["passengers"].isna().sum())

print(
    "Flights represented by rows with missing passengers:",
    international.loc[
        international["passengers"].isna(),
        "flights"
    ].sum()
)

Missing passenger values: 16
Flights represented by rows with missing passengers: 52


In [39]:
for column in ["airport_code", "flight_type", "month"]:
    print(column)
    print(international[column].value_counts(dropna=False))
    print()

airport_code
airport_code
DMM    1805
Name: count, dtype: int64

flight_type
flight_type
International    1805
Name: count, dtype: int64

month
month
Aug     176
Jun     175
Jul     175
Sep     168
May     163
Mar     159
Apr     159
Oct     158
Jan     157
Feb     155
Nov      78
Dec      78
Jan       4
Name: count, dtype: int64



In [40]:
international["month"] = international["month"].str.strip()

In [41]:
international["month"].value_counts(dropna=False)

month
Aug    176
Jun    175
Jul    175
Sep    168
May    163
Jan    161
Mar    159
Apr    159
Oct    158
Feb    155
Nov     78
Dec     78
Name: count, dtype: int64

## 6. Validate international dates and totals

Passenger sums use the available values, matching the published total.

In [42]:
international["month_number"] = international["month"].map(month_mapping)

international["date"] = pd.to_datetime(
    international["year"].astype(str) + "-" + international["month"],
    format="%Y-%b"
)

international["quarter"] = international["date"].dt.quarter

In [43]:
international[
    ["year", "month", "month_number", "date", "quarter"]
].drop_duplicates().sort_values(["year", "month_number"])

,year,month,month_number,date,quarter
862,2024,Jan,1,2024-01-01,1
863,2024,Feb,2,2024-02-01,1
864,2024,Mar,3,2024-03-01,1
865,2024,Apr,4,2024-04-01,2
866,2024,May,5,2024-05-01,2
867,2024,Jun,6,2024-06-01,2
868,2024,Jul,7,2024-07-01,3
869,2024,Aug,8,2024-08-01,3
870,2024,Sep,9,2024-09-01,3
871,2024,Oct,10,2024-10-01,4


In [44]:
international_source_total["Pax"] = (
    international_source_total["Pax"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .astype("int64")
)

international_source_total["ATMs"] = (
    international_source_total["ATMs"]
    .str.strip()
    .str.replace(",", "", regex=False)
    .astype("int64")
)

In [45]:
print("Calculated passengers:", international["passengers"].sum())
print(
    "Source passengers:",
    international_source_total["Pax"].iloc[0]
)

print("Calculated flights:", international["flights"].sum())
print(
    "Source flights:",
    international_source_total["ATMs"].iloc[0]
)

Calculated passengers: 11652478
Source passengers: 11652478
Calculated flights: 84828
Source flights: 84828


In [46]:
assert international["passengers"].sum() == international_source_total["Pax"].iloc[0]
assert international["flights"].sum() == international_source_total["ATMs"].iloc[0]

print("International source totals match.")

International source totals match.


In [47]:
international = international[
    [
        "airport_code",
        "date",
        "year",
        "quarter",
        "month_number",
        "month",
        "flight_type",
        "direction",
        "city",
        "passengers",
        "flights",
    ]
]

In [48]:
assert international.shape == (1805, 11)
assert international.duplicated(subset=key_columns).sum() == 0
assert international.drop(columns="passengers").isna().sum().sum() == 0
assert international["passengers"].isna().sum() == 16
assert (international["passengers"].dropna() > 0).all()
assert (international["flights"] > 0).all()
assert set(international["direction"]) == {"Arrival", "Departure"}
assert set(international["airport_code"]) == {"DMM"}
assert set(international["flight_type"]) == {"International"}
print("International validation passed.")

International validation passed.


## 7. Combine, validate, and export

One row represents an airport, reporting month, flight type, direction, and city.

In [49]:
traffic = pd.concat(
    [data, international],
    ignore_index=True
)

In [50]:
traffic["flight_type"].value_counts()

flight_type
International    1805
Domestic          674
Name: count, dtype: int64

In [51]:
combined_key = [
    "airport_code",
    "year",
    "month_number",
    "flight_type",
    "direction",
    "city",
]

assert traffic.shape == (2479, 11)
assert traffic.duplicated(subset=combined_key).sum() == 0
assert traffic["passengers"].isna().sum() == 16
assert traffic["flights"].isna().sum() == 0
assert (traffic["passengers"].dropna() > 0).all()
assert (traffic["flights"] > 0).all()

print("Combined traffic validation passed.")

Combined traffic validation passed.


In [52]:
assert traffic.drop(columns="passengers").isna().sum().sum() == 0

In [53]:
output_file = "../data/processed/dammam_airport_traffic_clean.csv"

traffic.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Saved {len(traffic)} rows to {output_file}")

Saved 2479 rows to ../data/processed/dammam_airport_traffic_clean.csv


In [54]:
traffic_check = pd.read_csv(
    "../data/processed/dammam_airport_traffic_clean.csv"
)

print(traffic_check.shape)
traffic_check.head()

(2479, 11)


,airport_code,date,year,quarter,month_number,month,flight_type,direction,city,passengers,flights
0,DMM,2025-01-01,2025,1,1,Jan,Domestic,Arrival,Abha,20359.0,132
1,DMM,2025-02-01,2025,1,2,Feb,Domestic,Arrival,Abha,17561.0,117
2,DMM,2025-03-01,2025,1,3,Mar,Domestic,Arrival,Abha,15262.0,122
3,DMM,2025-04-01,2025,2,4,Apr,Domestic,Arrival,Abha,23333.0,150
4,DMM,2025-05-01,2025,2,5,May,Domestic,Arrival,Abha,22267.0,162
